# Tema 11 — Segmentación y detección de bordes con OpenCV

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ecamposv/nlp-vision/blob/main/semana-06/Tema-11/Tema_11.ipynb)

Puedes ejecutar este notebook localmente (VS Code / Jupyter) o en **Google Colab** dando clic en el badge de arriba.

En este notebook exploramos distintas técnicas clásicas de visión computacional con **OpenCV**: detección de bordes (Sobel y Canny), búsqueda de contornos y varios métodos de segmentación (K-Means, Watershed y GrabCut).

**Carga de imagen con OpenCV**

### ¿Qué hace este código?

Detecta **bordes** usando el operador **Sobel**, que mide cómo cambia la intensidad de los píxeles:

1. **Helper `show_img`**: muestra imágenes con `matplotlib`, por lo que funciona tanto en local como en **Google Colab** (a diferencia de `cv.imshow`).
2. **Cargar la imagen** con `cv.imread`. Si la ruta `img_path` no existe (caso típico en Colab), se **descarga una imagen de ejemplo** automáticamente. Después se **valida** que la imagen se cargó (si es `None`, se lanza un error claro en lugar del fallo de `cvtColor`).
3. **Preproceso**: se convierte a escala de grises (`cvtColor`) y se suaviza con un desenfoque gaussiano (`GaussianBlur`) para reducir el ruido.
4. **Sobel**: calcula las derivadas en X (`gx`, cambios horizontales) y en Y (`gy`, cambios verticales). La **magnitud** del gradiente (`cv.magnitude`) indica la fuerza del borde.
5. **Normalización** a un rango 0–255 (`cv.normalize`) para poder visualizar los resultados como imágenes.
6. **Visualización** de cada resultado con `show_img`.

> 💡 Para usar **tu propia imagen** en Colab, súbela al entorno (panel de archivos) y ajusta `img_path`.

In [ ]:
import os
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt
import urllib.request

# ---- 0) Helper para mostrar imágenes (funciona en local y en Google Colab)
# cv.imshow abre ventanas del SO y NO funciona en Colab; usamos matplotlib.
def show_img(title, img, cmap=None):
    plt.figure(figsize=(6, 5))
    if img.ndim == 3:  # BGR -> RGB para matplotlib
        plt.imshow(cv.cvtColor(img, cv.COLOR_BGR2RGB))
    else:              # imagen en escala de grises
        plt.imshow(img, cmap=cmap or "gray")
    plt.title(title)
    plt.axis("off")
    plt.show()

# ---- 1) Cargar imagen
img_path = "images/imagen_1.png"   # <- cambia la ruta por tu imagen

# Si la imagen no existe (p. ej. en Colab), descargamos una de ejemplo
if not os.path.exists(img_path):
    os.makedirs(os.path.dirname(img_path), exist_ok=True)
    sample_url = "https://upload.wikimedia.org/wikipedia/commons/3/3f/Fronalpstock_big.jpg"
    print(f"'{img_path}' no encontrada. Descargando imagen de ejemplo...")
    urllib.request.urlretrieve(sample_url, img_path)

img = cv.imread(img_path)

# Validar que la imagen se cargó correctamente (evita el error de cvtColor)
if img is None:
    raise FileNotFoundError(
        f"No se pudo cargar la imagen en '{img_path}'. "
        "Verifica la ruta o sube una imagen al entorno."
    )

# ---- 2) Preproceso (gris + suavizado)
gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
blur = cv.GaussianBlur(gray, (5, 5), 0)

# ---- 3) Sobel (gradientes)
gx = cv.Sobel(blur, cv.CV_32F, 1, 0, ksize=3)  # derivada en X
gy = cv.Sobel(blur, cv.CV_32F, 0, 1, ksize=3)  # derivada en Y

# Magnitud del gradiente (fuerza del borde)
mag = cv.magnitude(gx, gy)

# ---- 4) Normalizar a 0-255 para visualizar
gx_u8 = cv.normalize(np.abs(gx), None, 0, 255, cv.NORM_MINMAX).astype(np.uint8)
gy_u8 = cv.normalize(np.abs(gy), None, 0, 255, cv.NORM_MINMAX).astype(np.uint8)
mag_u8 = cv.normalize(mag, None, 0, 255, cv.NORM_MINMAX).astype(np.uint8)

# ---- 5) Mostrar resultados
show_img("Original", img)
show_img("Gris", gray)
show_img("Sobel Gx (cambios horizontales)", gx_u8)
show_img("Sobel Gy (cambios verticales)", gy_u8)
show_img("Sobel Magnitud (bordes)", mag_u8)


**Detección de bordes con Canny**

### ¿Qué hace este código?

Detecta bordes con **Canny** y a partir de ellos encuentra los **contornos** de los objetos:

1. **Canny** (`cv.Canny`) produce una imagen binaria de bordes usando dos umbrales (50 y 150).
2. **Cierre morfológico** (`morphologyEx` con `MORPH_CLOSE`) rellena pequeños huecos para obtener contornos más completos.
3. **findContours** detecta los contornos externos (`RETR_EXTERNAL`) y comprime los puntos redundantes (`CHAIN_APPROX_SIMPLE`).
4. **drawContours** dibuja todos los contornos encontrados en verde sobre una copia de la imagen original.
5. Se imprime el número total de contornos y se muestran las imágenes intermedias.

In [ ]:
# ---- 6) Bordes (Canny) para tener una imagen binaria base
edges = cv.Canny(blur, 50, 150)

# (Opcional) Cerrar huecos para contornos más completos
kernel = np.ones((3, 3), np.uint8)
edges_closed = cv.morphologyEx(edges, cv.MORPH_CLOSE, kernel, iterations=1)

# ---- 4) Encontrar contornos
# RETR_EXTERNAL: solo contornos externos
# CHAIN_APPROX_SIMPLE: comprime puntos redundantes
contours, hierarchy = cv.findContours(edges_closed, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)

# ---- 5) Dibujar contornos
out = img.copy()
cv.drawContours(out, contours, -1, (0, 255, 0), 2)  # -1 = todos, color verde, grosor 2

# ---- 6) Mostrar resultados
print(f"Total de contornos encontrados: {len(contours)}")

cv.imshow("Original", img)
cv.imshow("Edges (Canny)", edges)
cv.imshow("Edges Closed (morfologia)", edges_closed)
cv.imshow("Contornos (drawContours)", out)

**Segmentación de objetos con K-Means**

### ¿Qué hace este código?

Segmenta la imagen agrupando píxeles de color similar con el algoritmo **K-Means**:

1. Se convierte la imagen al espacio de color **Lab** (`COLOR_BGR2LAB`), más estable para comparar colores.
2. Los píxeles se reorganizan en una matriz de `N × 3` (`reshape`) como entrada para K-Means.
3. **`cv.kmeans`** agrupa los píxeles en `K` clusters (aquí `K = 3`). Cada píxel se asigna al centro más cercano.
4. Se **reconstruye** la imagen segmentada reemplazando cada píxel por el color de su cluster.
5. Se generan **máscaras** binarias por cada cluster y se guardan con `cv.imwrite`, útiles para ver qué región capturó cada grupo.

> 💡 Prueba distintos valores de `K` (2, 3, 4) según la cantidad de regiones que quieras separar.
> ⚠️ La variable `OUT_DIR` debe estar definida antes de ejecutar (carpeta donde se guardan las máscaras).

In [ ]:
import os
import cv2 as cv
import numpy as np

img_path = "images/imagen_2.png"  # <- cambia la ruta
img = cv.imread(img_path)
# ---- 1) Convertir a un espacio de color más estable (Lab)
lab = cv.cvtColor(img, cv.COLOR_BGR2LAB)

# ---- 2) Preparar datos para K-means: (N_pixeles x 3)
Z = lab.reshape((-1, 3)).astype(np.float32)

# ---- 3) Ejecutar K-means
K = 3  # prueba 2, 3, 4 según tu imagen
criteria = (cv.TERM_CRITERIA_EPS + cv.TERM_CRITERIA_MAX_ITER, 20, 1.0)
attempts = 5
flags = cv.KMEANS_PP_CENTERS
compactness, labels, centers = cv.kmeans(Z, K, None, criteria, attempts, flags)

# ---- 4) Reconstruir imagen segmentada
centers = np.uint8(centers)
seg_lab = centers[labels.flatten()].reshape(lab.shape)
seg_bgr = cv.cvtColor(seg_lab, cv.COLOR_LAB2BGR)

# ---- 5) (Opcional) Visualizar las K "máscaras" de cada cluster
h, w = lab.shape[:2]
labels_2d = labels.reshape((h, w))

masks = []
for i in range(K):
    mask = np.where(labels_2d == i, 255, 0).astype(np.uint8)
    masks.append(mask)
    cv.imwrite(os.path.join(OUT_DIR, f"kmeans_mask_cluster_{i}.png"), mask)

# ---- 7) Mostrar resultados
cv.imshow("K-means - 00 Original", img)
cv.imshow("K-means - 01 Segmented", seg_bgr)

# Mostrar máscaras por cluster (útil para ver qué región capturó cada grupo)
for i in range(K):
    cv.imshow(f"K-means - Mask Cluster {i}", masks[i])
cv.waitKey(0)
cv.destroyAllWindows()

**Detección de bordes con Watershed**

### ¿Qué hace este código?

Separa objetos que se tocan entre sí usando el algoritmo **Watershed** (cuencas hidrográficas):

1. **Preproceso**: escala de grises + suavizado.
2. **Binarización con Otsu** (`THRESH_OTSU`) calcula automáticamente el umbral. Usa `THRESH_BINARY_INV` si los objetos son claros sobre fondo oscuro.
3. **Apertura morfológica** (`MORPH_OPEN`) elimina ruido pequeño.
4. **Fondo seguro** (`sure_bg`): se obtiene dilatando el objeto.
5. **Objeto seguro** (`sure_fg`): se calcula con la **transformada de distancia** (`distanceTransform`) y un umbral; marca los centros de los objetos.
6. **Región desconocida**: la diferencia entre fondo y objeto seguro (las fronteras dudosas).
7. **Marcadores**: se etiquetan los objetos con `connectedComponents` y se ejecuta **`cv.watershed`**, que dibuja las fronteras (`markers == -1`) en rojo.

> ⚠️ Ajusta `fg_ratio` (0.35) para controlar qué tan estrictos son los "objetos seguros".

In [ ]:
import os
import cv2 as cv
import numpy as np

img_path = "images/imagen_5.png"  # <- cambia la ruta
img = cv.imread(img_path)
# ---- 1) Preproceso: gris + suavizado
gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
blur = cv.GaussianBlur(gray, (5, 5), 0)

# ---- 2) Binarización (Otsu)
# IMPORTANTE:
# - Si tus OBJETOS son CLAROS sobre fondo OSCURO -> usa THRESH_BINARY_INV
# - Si tus OBJETOS son OSCUROS sobre fondo CLARO -> usa THRESH_BINARY
_, thresh = cv.threshold(blur, 0, 255, cv.THRESH_BINARY_INV + cv.THRESH_OTSU)

# ---- 3) Limpieza (apertura morfológica)
kernel = np.ones((3, 3), np.uint8)
opening = cv.morphologyEx(thresh, cv.MORPH_OPEN, kernel, iterations=2)

# ---- 4) Fondo seguro (dilatación)
sure_bg = cv.dilate(opening, kernel, iterations=3)

# ---- 5) Objeto seguro (distance transform + umbral)
dist = cv.distanceTransform(opening, cv.DIST_L2, 5)

fg_ratio = 0.35
_, sure_fg = cv.threshold(dist, fg_ratio * dist.max(), 255, 0)
sure_fg = np.uint8(sure_fg)

# ---- 6) Región desconocida
unknown = cv.subtract(sure_bg, sure_fg)

# ---- 7) Marcadores
n_labels, markers = cv.connectedComponents(sure_fg)
markers = markers + 1            # para que el fondo no sea 0
markers[unknown == 255] = 0      # zona desconocida = 0

# ---- 8) Watershed
markers = cv.watershed(img, markers)

# ---- 9) Visualización: fronteras en rojo donde markers == -1
out = img.copy()
out[markers == -1] = (0, 0, 255)

# ---- 10) Normalizar mapas para visualizar/guardar
dist_norm = cv.normalize(dist, None, 0, 255, cv.NORM_MINMAX).astype(np.uint8)

# visualizar markers como imagen (solo para inspección)
# convertimos markers a rango 0..255 para verlo
markers_vis = markers.copy()
markers_vis[markers_vis < 0] = 0
markers_vis = cv.normalize(markers_vis.astype(np.float32), None, 0, 255, cv.NORM_MINMAX).astype(np.uint8)

**Segmentación de objetos en imágenes con GrabCut**

### ¿Qué hace este código?

Extrae el objeto principal del fondo con **GrabCut**, un método interactivo basado en un rectángulo inicial:

1. Se define un **rectángulo** (`rect`) que encierra el objeto de interés (aquí el 80 % central de la imagen).
2. Se inicializan la **máscara** y los modelos internos (`bgdModel`, `fgdModel`) que OpenCV necesita.
3. **`cv.grabCut`** se ejecuta con `GC_INIT_WITH_RECT` durante varias iteraciones, estimando qué es fondo y qué es primer plano.
4. La máscara resultante se convierte a **binaria** (0 = fondo, 1 = objeto) combinando los valores seguros y probables.
5. Se aplica la máscara a la imagen para **extraer el objeto**, dejando el fondo en negro.

> 💡 Si el objeto queda recortado o entra demasiado fondo, ajusta el `rect` o el número de iteraciones (`iters`).

In [ ]:
import os
import cv2 as cv
import numpy as np

img_path = "images/imagen_3.png"  # <- cambia la ruta
img = cv.imread(img_path)
h, w = img.shape[:2]
# ---- 1) Definir rectángulo inicial (x, y, width, height)
# Ajusta estos valores si el objeto queda recortado o si entra demasiado fondo
rect = (int(0.10 * w), int(0.10 * h), int(0.80 * w), int(0.80 * h))

# (Opcional) Visualizar el rectángulo sobre la imagen
rect_vis = img.copy()
x, y, rw, rh = rect
cv.rectangle(rect_vis, (x, y), (x + rw, y + rh), (0, 255, 255), 2)

# ---- 2) Inicializar máscara y modelos internos requeridos por OpenCV
mask = np.zeros((h, w), np.uint8)            # 0 al inicio (se llena durante GrabCut)
bgdModel = np.zeros((1, 65), np.float64)     # requerido por OpenCV
fgdModel = np.zeros((1, 65), np.float64)     # requerido por OpenCV

# ---- 3) Ejecutar GrabCut
iters = 5  # prueba 3, 5, 7
cv.grabCut(img, mask, rect, bgdModel, fgdModel, iters, cv.GC_INIT_WITH_RECT)

# ---- 4) Convertir la máscara a binaria (0=fondo, 1=primer plano)
# mask valores:
# 0 = background seguro
# 1 = foreground seguro
# 2 = background probable
# 3 = foreground probable
mask_bin = np.where((mask == 0) | (mask == 2), 0, 1).astype("uint8")
mask_vis = (mask_bin * 255).astype(np.uint8)

# ---- 5) Aplicar la máscara para extraer el objeto
result = img * mask_bin[:, :, None]

# ---- 7) Mostrar resultados
cv.imshow("GrabCut - 00 Original", img)
cv.imshow("GrabCut - 01 Rectangulo inicial", rect_vis)
cv.imshow("GrabCut - 02 Mascara binaria", mask_vis)
cv.imshow("GrabCut - 03 Resultado (objeto extraido)", result)
cv.waitKey(0)
cv.destroyAllWindows()